In [1]:
# Pre-training SoRL on arithmatic generalization dataset
# ------------------------------------------------------ 
import torch
from sorl.gat_sim import GAT, GATConfig
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup

BOS_TOKEN_ID = 20
gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 16],  # 16 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu",
    # bos_token_id=BOS_TOKEN_ID
)
    
model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

Generate multiplication data: 

```python data/arithmetic.py```

In [3]:
# ---- Arithmatic generalization dataset loader ----
from sorl.arithmetic import data_generator
from sorl.arithmetic import DigitTokenizer

# --- tokenizer ---
tokenizer = DigitTokenizer()

# --- data loader ---
train_loader = data_generator(filename_pattern="data/multiplication/multiplication_train.bin", sequence_length=256, device="cpu")
val_loader = data_generator(filename_pattern="data/multiplication/multiplication_val_id.bin", sequence_length=64, device="cpu")

In [8]:
# Information Gain Fromulation
# ------------------------------------------------------------------------
from sorl.neo_utils import sorl_evaluate, sorl_search_v8
from sorl.topo import orthogonalize_abs_param
from collections import defaultdict
from sorl.info import SoRLLoss_v7

# --- orthogonal initialization on abs param --- 
orthogonalize_abs_param(model, do_wte=True, do_head=True)

K = 4
max_iterations = 2
loss_fn = SoRLLoss_v7(model.vocab_sizes[1], decay=0.8, target_vocab_util=0.9)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
n = 2
temperature = torch.tensor([0.0, 5.0], device=model.device)
num_steps = 400
alpha_abs = 0.1
alpha_soft_zipf = 1.0
alpha_info_gain = 10.0
attn_blocksize = 1792
phase = "compression"
memory_span = 1792 

record = defaultdict(list)
img_frames = []

for step in range(num_steps): 

    optimizer.zero_grad()

    tokens = next(train_loader)

    with torch.no_grad(): 
        best_data, best_traj_ppt, best_abs_ppt, search_adv = sorl_search_v8(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperature, truncate_seq_len=False)

    # --- compute loss --- 
    base_traj_loss, base_logits = model.forward(tokens, memory_span, attn_blocksize)
    base_traj_loss = base_traj_loss.mean()
    info_gain_loss, abs_loss, zipf_bigram_loss = loss_fn(best_data, model, base_traj_loss.detach(), memory_span, attn_blocksize)
    loss = base_traj_loss + alpha_info_gain * info_gain_loss.mean() + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss

    # --- log relative info gain ---
    rel_info_gain = ((-info_gain_loss) / base_traj_loss).detach()

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            temperatures_eval = torch.tensor([0.0, 10.0], device=model.device)

            val_tokens, val_adv, traj_loss, abs_loss = sorl_evaluate(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                     truncate_seq_len=False)
            # _, _, zipf_bigram_loss = loss_fn(val_tokens, model, memory_span_abs, memory_span_traj, attn_blocksize)
     

        # print(f"\n{phase} | step {step} | base traj loss: {base_traj_loss.item():.2f} | cond traj loss: {traj_loss.mean().item():.2f} | rel search info gain: {rel_info_gain * 100:.2f}% | greedy adv: {val_adv.item() * 100:.2f}% | vocab util: {abs_stats.vocab_util * 100:.2f}%  | avg logit sim: {avg_logit_sim:.2f} |  bigram-zipf kl: {zipf_bigram_loss.item():.2f} | bigram rep rate: {abs_stats.bigram_rep_rate:.2f} | rel info gain (search): {rel_info_gain:.2f}")
        print("Step", step)
        # img = visualize_dynamics(abs_stats, loader, model, enc, K, step)
        # img_frames.append(img)
        # break

Step 0
Step 2
Step 4
Step 6
Step 8
Step 10
Step 12
Step 14
Step 16
Step 18
Step 20
Step 22
Step 24
Step 26
Step 28
Step 30
Step 32
Step 34
Step 36
Step 38
Step 40
Step 42
Step 44
Step 46
Step 48
Step 50
Step 52
Step 54
Step 56
Step 58
Step 60
Step 62
Step 64
Step 66
Step 68
Step 70
Step 72
Step 74
Step 76
Step 78
Step 80
Step 82
Step 84
Step 86
Step 88
Step 90
Step 92
Step 94
Step 96
Step 98
Step 100
Step 102
Step 104
Step 106
Step 108
Step 110
Step 112
Step 114
Step 116
Step 118
Step 120
Step 122
Step 124
Step 126
Step 128
Step 130
Step 132
Step 134
Step 136
Step 138
Step 140
Step 142
Step 144
Step 146
Step 148
Step 150
Step 152
Step 154
Step 156
Step 158
Step 160
Step 162
Step 164
Step 166
Step 168
Step 170
Step 172
Step 174
Step 176
Step 178
Step 180
Step 182
Step 184
Step 186
Step 188
Step 190
Step 192
Step 194
Step 196
Step 198
Step 200
Step 202
Step 204
Step 206
Step 208
Step 210
Step 212
Step 214
Step 216
Step 218
Step 220
Step 222
Step 224
Step 226
Step 228
Step 230
Step 232
St

In [9]:
val_tokens

tensor([[ 0, 18,  4,  2, 24,  4, 11, 11, 17, 22,  4,  3,  4, 19, 25, 13, 16,  1,
          0, 23, 18, 15, 19,  4, 24,  2,  4, 13, 14, 22,  4,  3,  4, 12, 25, 19,
         12, 10, 16, 30,  1,  0, 18, 12, 22, 19,  4,  2,  4, 27, 16, 11, 10,  4,
         28,  3,  4, 15, 10, 25, 15, 16, 19, 10, 30,  1,  0, 19, 16, 22, 10,  4,
          2,  4, 27, 15,  4,  3,  4, 28, 14, 18, 10, 10, 30,  1,  0, 16,  4, 24,
          2,  4, 17, 10, 22,  4,  3,  4, 14, 25, 12, 10,  1,  0, 23, 13, 15,  4,
          2, 24,  4, 16,  4,  3, 22,  4, 12, 11, 10, 30,  1,  0, 11, 13, 22,  4,
          2,  4, 12, 22, 13,  4,  3,  4, 28, 12, 19, 19,  1, 29,  0, 19, 14,  4,
         24,  2,  4, 13,  4, 23,  3,  4, 12, 18, 25, 12,  1,  0, 19, 22, 17, 12,
          4,  2, 24,  4, 17, 14,  4, 28,  3,  4, 17, 11, 25, 19, 12, 18,  1, 29,
          0, 15,  4,  2, 24,  4, 18,  4,  3, 22,  4, 14, 10,  1, 29,  0, 11, 14,
         17, 22,  4,  2,  4, 15, 22, 11,  4,  3,  4, 28, 17, 14, 19, 17, 25,  1,
          0, 19, 10, 22,  4,

In [11]:
# generate function implementation 
# ----------------------------------
from sorl.neo_utils import generate
from sorl.arithmetic import process_query, check_answer

K = 5
tokens = next(val_loader)
idx, answer_idx = process_query(tokens)

print(f"init   | idx: {idx[0].tolist()} | question: {tokenizer.decode(idx[0].tolist()[1:-1])}")
for i in range(len(answer_idx[0])*2): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, 
                   temperature=temperatures_eval[0])
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    print(f"step {i+1} idx (abstraction free): {idx_without_abstraction.tolist()}")
    print(f"                         idx : {idx[0].tolist()}")

is_correct, pred_answer, true_answer = check_answer(idx_without_abstraction, answer_idx, tokenizer)
print(f"is_correct: {is_correct} | pred_answer: {pred_answer} | true_answer: {true_answer}")

init   | idx: [0, 15, 4, 2, 4, 18, 4, 3] | question: 5 x 8 
step 1 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4]
step 2 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11]
step 3 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11, 16]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11, 16]
step 4 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11, 16, 1]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11, 25, 16, 1]
step 5 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11, 16, 1, 0]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11, 25, 16, 1, 0]
step 6 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11, 16, 1, 0, 17]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11, 25, 16, 1, 0, 17]
step 7 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 1